In [0]:
%sql
-- Filter rows with usable weather
CREATE OR REPLACE TABLE default.model_dataset AS
SELECT *
FROM default.gold_county_yield_weather
WHERE prcp_total_mm IS NOT NULL
  AND tmax_avg_c IS NOT NULL
  AND tmin_avg_c IS NOT NULL;

In [0]:
%sql
-- sanity check
SELECT COUNT(*) FROM default.model_dataset;

In [0]:
# Switch to Python (Spark ML) Gradient Boosted Trees (strong + explainable)
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline

df = spark.table("default.model_dataset")

# Encode categorical variables
commodity_indexer = StringIndexer(inputCol="commodity", outputCol="commodity_idx")
irrigation_indexer = StringIndexer(inputCol="irrigation", outputCol="irrigation_idx")

assembler = VectorAssembler(
    inputCols=[
        "prcp_total_mm",
        "tmax_avg_c",
        "tmin_avg_c",
        "heat_days_30c",
        "year",
        "commodity_idx",
        "irrigation_idx"
    ],
    outputCol="features"
)

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="yield",
    maxIter=50,
    maxDepth=5
)

pipeline = Pipeline(stages=[
    commodity_indexer,
    irrigation_indexer,
    assembler,
    gbt
])

model = pipeline.fit(df)
predictions = model.transform(df)

predictions.createOrReplaceTempView("yield_predictions")

In [0]:
%sql
-- Compute Residual Anomalies
CREATE OR REPLACE TABLE default.yield_residual_anomalies AS
SELECT
  *,
  yield - prediction AS residual
FROM yield_predictions;

In [0]:
%sql
-- Find biggest shocks
SELECT
  county_fips,
  year,
  commodity,
  yield,
  prediction,
  residual
FROM default.yield_residual_anomalies
ORDER BY ABS(residual) DESC
LIMIT 20;

In [0]:
# Explain What Drives Yield
gbt_model = model.stages[-1]

import pandas as pd

feature_importance = pd.DataFrame({
    "feature": [
        "prcp_total_mm",
        "tmax_avg_c",
        "tmin_avg_c",
        "heat_days_30c",
        "year",
        "commodity_idx",
        "irrigation_idx"
    ],
    "importance": gbt_model.featureImportances.toArray()
})

feature_importance.sort_values("importance", ascending=False)